# 15 — Observability, Metrics, and SLOs

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from datetime import datetime, timezone
import math

# Provided for this lesson.
from collections.abc import Callable

from lesson import Exercises, ___, run

check = Exercises('advanced/15', {1: '6cd7d90800817e5d', 2: '8b44641620759b90', 3: '9adee7d3d49d42ce', 4: '0c317ed4acd3b380', 5: 'f2aa46b1e22e549d', 6: '6fdfe127cc16757e', 7: 'db2b86a5de5024c9', 8: 'd430ce7f51cb1380', 9: '68a6fbabcdcad028'}, practice='advanced_04_production_systems', independent='availability')

## 1. Structured events

- Log events as data (a dict, written as JSON), not free text: `{"code": "order.paid", "correlation_id": ..., "timestamp": ...}`.
- A correlation id follows one request through every service; generate one with `uuid.uuid4()` if none came in.
- Timestamps in UTC ISO-8601: `datetime.now(timezone.utc).isoformat()`. Inject the clock for tests.

In [ ]:
import json, uuid
from datetime import datetime, timezone
fixed = lambda: datetime(2024, 5, 1, 9, 30, tzinfo=timezone.utc)
event = {"version": 1, "code": "order.paid", "correlation_id": str(uuid.uuid4()), "timestamp": fixed().isoformat()}
print(json.dumps(event))

**Exercise 1.** Format a fixed UTC time as ISO-8601.

In [ ]:
from datetime import datetime, timezone
when = datetime(2024, 1, 2, 3, 4, tzinfo=timezone.utc)
text = when.isoformat()
predicted = ___
check(1, predicted)

**Exercise 2.** Keep an incoming correlation id; generate one only when it's missing.

In [ ]:
import uuid
def correlation(incoming):
    return incoming if incoming is not None else str(uuid.uuid4())

kept = correlation(___)
check(2, kept)   # pass "req-42" through

**Exercise 3.** How long is a `str(uuid.uuid4())`?

In [ ]:
import uuid
predicted = ___
check(3, predicted)

### 1. `make_event`

A versioned event with a correlation id and a UTC timestamp.

In [ ]:
def make_event(code: str, correlation_id: str | None = None,
               clock: Callable[[], datetime] = lambda: datetime.now(timezone.utc)) -> dict:
    """Return {'version': 1, 'code', 'correlation_id', 'timestamp'} with an ISO-8601 UTC timestamp.

    A missing correlation_id is generated with uuid4. The clock is injectable for tests.
    """
    return ___

In [ ]:
check('make_event')

## 2. Latency and availability

- Averages hide slow requests. Report the median and a high percentile like p95.
- Nearest-rank p95: sort, then take the value at 1-based position `ceil(0.95 * n)`.
- Availability = successes / total × 100. With no traffic, report 100.0.

In [ ]:
import math, statistics
ms = [12, 15, 11, 300, 14, 13, 16, 12, 15, 14]
ordered = sorted(ms)
rank = math.ceil(0.95 * len(ordered))
print(statistics.mean(ms), statistics.median(ms), ordered[rank - 1])
print(round(100 * 9_990 / 10_000, 3))

**Exercise 4.** The nearest-rank p95 position for 20 samples (1-based).

In [ ]:
import math
rank = math.ceil(0.95 * ___)
check(4, rank)

**Exercise 5.** p95 of `range(1, 21)`.

In [ ]:
import math
ordered = list(range(1, 21))
p95 = ordered[math.ceil(0.95 * len(ordered)) - ___]
check(5, p95)

**Exercise 6.** Availability of 997 successes out of 1000, as a percentage.

In [ ]:
availability = round(100 * 997 / ___, 3)
check(6, availability)

## 3. SLOs, error budgets, alerts

- An SLO is a target, like 99.9% of requests succeed. The error budget is what's left: `requests × (100 − target) / 100`, rounded down.
- Alert on sustained problems (the last few samples all bad), not single spikes, to avoid flapping.
- A readiness check runs every probe; a probe that raises counts as failed.

In [ ]:
import math
print(math.floor(1_000_000 * (100 - 99.9) / 100 + 1e-9))
rates = [0.01, 0.2, 0.3, 0.25]
print(all(r > 0.1 for r in rates[-3:]))

**Exercise 7.** Error budget for 99% over 5,000 requests.

In [ ]:
import math
budget = math.floor(5000 * (100 - ___) / 100)
check(7, budget)

**Exercise 8.** Alert only if the last 3 rates are all above 0.05.

In [ ]:
rates = [0.2, 0.01, 0.06, 0.07, 0.09]
alert = all(r > 0.05 for r in rates[___:])
check(8, alert)

**Exercise 9.** Run the probes; one raises. Predict the results.

In [ ]:
def broken():
    raise ConnectionError
checks = {"db": lambda: True, "cache": broken}
results = {}
for name, probe in checks.items():
    try:
        results[name] = "ok" if probe() else "fail"
    except Exception:
        results[name] = "fail"
predicted = ___
check(9, predicted)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 2. `latency_summary`

Count, median and nearest-rank p95.

In [ ]:
def latency_summary(milliseconds: list[float]) -> dict[str, float]:
    """Return {'count', 'median', 'p95'}; p95 uses the nearest-rank method.

    nearest rank: sort, then take the value at position ceil(0.95 * n) (1-based).
    Raise ValueError for an empty list.
    """
    if ___:
        raise ValueError("no samples")
    ordered = sorted(milliseconds)
    rank = math.ceil(0.95 * len(ordered))
    return ___

In [ ]:
check('latency_summary')

### Independent build — `availability`

Return successes as a percentage of total, rounded to three decimal places. An interval with no requests reports 100.0. Assume 0 ≤ successes ≤ total.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def availability(successes: int, total: int) -> float:
    raise NotImplementedError

In [ ]:
check('availability')

### 4. `error_budget`

How many requests may fail while meeting the target.

In [ ]:
def error_budget(target_percent: float, requests: int) -> int:
    """How many requests may fail while still meeting the target (round down).

    error_budget(99.9, 1_000_000) == 1000
    """
    return ___

In [ ]:
check('error_budget')

### 5. `should_alert`

Alert only when the last samples are all above threshold.

In [ ]:
def should_alert(error_rates: list[float], threshold: float, sustained: int) -> bool:
    """Alert only if the last `sustained` samples are ALL above threshold (avoids flapping)."""
    raise NotImplementedError

In [ ]:
check('should_alert')

### 6. `readiness`

Run every check; report readiness and each result.

In [ ]:
def readiness(checks: dict[str, Callable[[], bool]]) -> dict:
    """Run every check; a check that raises counts as failed. Return
    {'ready': all passed, 'checks': {name: 'ok' | 'fail'}}."""
    raise NotImplementedError

In [ ]:
check('readiness')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** An SLO is…

   - **a)** A target for a metric (e.g. 99.9% success)
   - **b)** A measured metric
   - **c)** An alert

**2.** A correlation ID lets you…

   - **a)** Follow one request across logs and services
   - **b)** Encrypt logs
   - **c)** Count users

**3.** Good alerts fire on…

   - **a)** Every single error
   - **b)** CPU spikes only
   - **c)** Sustained user-visible problems

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/15', {1: '20feb57c6faf', 2: 'b86084f78462', 3: 'aa34ad9c656e'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [16 — API Contracts, Versioning, Pagination, and Evolution](16_api_design_versioning_contracts.ipynb)